# 05c — Uncapacitated P-Median Benchmark (Greater London)

**Third of three companion notebooks** (alongside 05_p_median_715.ipynb / joint MILP+slack,
and 05b_two_stage_model.ipynb / two-stage capacity allocation). This one is the classic
textbook p-median: no eⱼ, no K, no capacity constraint at all -- p sites are opened,
every LSOA is assigned to its nearest open site, done. Matches the original proposal
formulation and the P012 (He, Kuo & Wu 2016) PMP-vs-MCLP comparison directly.

Because there is no capacity constraint, this MILP is always feasible for any 1<=p<=n
and k-NN pruning here is purely a speed optimisation, not a feasibility requirement
(unlike the capacitated case) -- k=20-30 (the proposal's originally declared range)
is used directly, with no need for hub-escape arcs or K margins.

## 0. Setup and reload cleaned data

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import pulp
from scipy.spatial import cKDTree
from scipy.stats import pearsonr
import time
import os

BASE = "/Users/alexia/Documents/CASA/Dissertation"

demand_london = pd.read_csv(os.path.join(BASE, "05_processed/demand_london.csv"))
imd_london    = pd.read_csv(os.path.join(BASE, "05_processed/imd_london_clean.csv"))
census_london = pd.read_csv(os.path.join(BASE, "05_processed/census_london_clean.csv"))

outputs_dir = os.path.join(BASE, "06_outputs")
os.makedirs(outputs_dir, exist_ok=True)

print("Datasets reloaded.")
print(f"demand_london: {demand_london.shape}, imd_london: {imd_london.shape}")

## 1. LSOA centroids (I = J) + fixed weights Vi and income_decile for M1-M4

In [ ]:
lsoa_boundaries = gpd.read_file(os.path.join(BASE, "03_data/demand/spatial/LSOA_2021_EW_BGC_V5.shp"))
if lsoa_boundaries.crs is None or lsoa_boundaries.crs.to_epsg() != 27700:
    lsoa_boundaries = lsoa_boundaries.to_crs(epsg=27700)

london_codes = set(demand_london["lsoa_code"])
lsoa_london = lsoa_boundaries[lsoa_boundaries["LSOA21CD"].isin(london_codes)].copy()
lsoa_london = lsoa_london.rename(columns={"LSOA21CD": "lsoa_code"})[["lsoa_code", "geometry"]]

lsoa_london["centroid"] = lsoa_london.geometry.centroid
lsoa_london["cx"] = lsoa_london["centroid"].x
lsoa_london["cy"] = lsoa_london["centroid"].y

lsoa_master = lsoa_london[["lsoa_code", "cx", "cy"]].merge(
    demand_london[["lsoa_code", "D_A", "D_B", "D_C", "D_D"]], on="lsoa_code", how="inner"
).merge(
    census_london[["lsoa_code", "Hi", "Ci"]], on="lsoa_code", how="left"
).merge(
    imd_london[["lsoa_code", "income_score", "income_decile"]], on="lsoa_code", how="left"
).reset_index(drop=True)
lsoa_master["Vi"] = lsoa_master["Hi"] * lsoa_master["Ci"]

n = len(lsoa_master)
coords = lsoa_master[["cx", "cy"]].to_numpy()
print(f"LSOA master table: {n} LSOAs (should be ~4,994)")
lsoa_master.head()

## 2. k-nearest-neighbour pruning

No capacity constraint, but pruning can still cause Infeasible: with p small relative to
n (p=100 is ~2% of ~4,994), a demand node's k-nearest candidates can by chance include
none of the sites that end up opened, leaving it with no valid assignment. k=60 (found
empirically, see code comment) avoids this at all three tested p values.

In [ ]:
# Raised from 25 to 60 after hitting Infeasible at p=100 (found empirically, confirmed
# on synthetic data at the same ~2% p/n density): with a small p relative to n, a demand
# node's own k-nearest candidates can, by chance, contain none of the eventually-opened
# sites -- every yij <= zj is then 0, so sum_j yij = 1 cannot be satisfied. This is
# unrelated to capacity; it is a pure candidate-pruning-vs-sparsity effect, worse at
# small p (p=100 is the sparsest of the three tested values). k=60 gives a comfortable
# margin above the k=50 threshold found in testing.
K_NEIGHBOURS = 60

tree = cKDTree(coords)
dist, idx = tree.query(coords, k=K_NEIGHBOURS + 1)

pair_i, pair_j, pair_d = [], [], []
for i in range(n):
    for rank in range(idx.shape[1]):
        pair_i.append(i)
        pair_j.append(int(idx[i, rank]))
        pair_d.append(float(dist[i, rank]))

candidate_pairs = pd.DataFrame({"i": pair_i, "j": pair_j, "dij": pair_d}).drop_duplicates(subset=["i", "j"])
print(f"Total (i,j) candidate pairs kept: {len(candidate_pairs):,} out of a possible {n*n:,}")
print(f"Average candidates per demand node: {len(candidate_pairs) / n:.1f}")

## 3. Core solver: classic uncapacitated p-median MILP

zj = 1 if LSOA j is opened as one of the p sites; yij = 1 if LSOA i is served by j.
No eⱼ, no K -- min demand-weighted distance, subject to exactly p sites open and every
LSOA assigned to an open site.

In [ ]:
def solve_uncapacitated_p_median(demand_col, p, candidate_pairs, lsoa_master,
                                  time_limit_sec=600, gap_rel=0.02, msg=False):
    """
    Classic p-median: min sum Di*dij*yij s.t. sum_j yij=1, yij<=zj, sum_j zj=p.
    Always feasible for any 1<=p<=n (no capacity constraint to violate).

    yij is relaxed to continuous [0,1] (only zj is binary). For uncapacitated p-median
    this is a standard, well-established simplification: given any fixed set of open
    sites, the assignment sub-problem is totally unimodular, so the LP relaxation of y
    already yields an integral (0/1) optimum -- relaxing it does not change the solution,
    but avoids branching on hundreds of thousands of y-variables, which is what caused
    'Not Solved' (time-limit, not infeasibility) at n~5000 with binary y.
    """
    t0 = time.time()
    n_ = len(lsoa_master)
    Di = lsoa_master[demand_col].to_numpy()

    prob = pulp.LpProblem("uncapacitated_p_median", pulp.LpMinimize)
    z = {j: pulp.LpVariable(f"z_{j}", cat="Binary") for j in range(n_)}
    y = {(row.i, row.j): pulp.LpVariable(f"y_{row.i}_{row.j}", lowBound=0, upBound=1)
         for row in candidate_pairs.itertuples()}

    prob += pulp.lpSum(Di[row.i] * row.dij * y[(row.i, row.j)] for row in candidate_pairs.itertuples())

    for i, group in candidate_pairs.groupby("i"):
        prob += pulp.lpSum(y[(i, j)] for j in group["j"]) == 1
        for j in group["j"]:
            prob += y[(i, j)] <= z[j]

    prob += pulp.lpSum(z[j] for j in range(n_)) == p

    solver = pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit_sec, gapRel=gap_rel)
    status_code = prob.solve(solver)
    status = pulp.LpStatus[status_code]
    runtime = time.time() - t0

    if status != "Optimal":
        print(f"  [solve_uncapacitated_p_median] status = {status} (p={p}) — NOT trusting variable values.")
        return {"status": status, "objective": None, "zj": None, "assigned_j": None,
                "dist": None, "runtime_sec": runtime}

    zj = np.array([int(round(z[j].value())) for j in range(n_)])
    assigned_j = np.full(n_, -1)
    dist = np.zeros(n_)
    for row in candidate_pairs.itertuples():
        if y[(row.i, row.j)].value() > 0.5:
            assigned_j[row.i] = row.j
            dist[row.i] = row.dij

    return {
        "status": status,
        "objective": pulp.value(prob.objective),
        "zj": zj,
        "assigned_j": assigned_j,
        "dist": dist,
        "runtime_sec": runtime,
    }


def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n_ = len(x)
    if x.sum() == 0:
        return 0.0
    cum = np.cumsum(x)
    return (n_ + 1 - 2 * (cum.sum() / cum[-1])) / n_


def compute_metrics(dist, lsoa_master):
    """M1-M4 exactly as defined in proposal Section 5.5, from a completed assignment's dist array."""
    Vi = lsoa_master["Vi"].to_numpy()
    decile = lsoa_master["income_decile"].to_numpy()

    M1 = float((Vi * dist).sum() / Vi.sum())
    within_800 = (dist < 800).astype(float)
    M2 = float((Vi * within_800).sum() / Vi.sum())

    def coverage_for_decile(d):
        mask = decile == d
        if mask.sum() == 0 or Vi[mask].sum() == 0:
            return np.nan
        return float((Vi[mask] * within_800[mask]).sum() / Vi[mask].sum())

    cov_d1, cov_d10 = coverage_for_decile(1), coverage_for_decile(10)
    M3 = cov_d1 - cov_d10

    eps = 1.0
    accessibility = 1.0 / (dist + eps)
    M4 = gini(accessibility)

    return {"M1_avg_dist_m": M1, "M2_coverage_800m": M2, "M3_imd_gap": M3, "M4_gini": M4}

## 4. Run the core 12-configuration grid (4 alpha scenarios x 3 p values)

In [ ]:
ALPHA_LABELS = {"A": "D_A", "B": "D_B", "C": "D_C", "D": "D_D"}
P_VALUES = [100, 250, 500]
TIME_LIMIT_SEC = 600

core_results = {}
core_summary_rows = []

for alpha_label, demand_col in ALPHA_LABELS.items():
    for p in P_VALUES:
        key = (alpha_label, p)
        print(f"=== Solving: Scenario {alpha_label}, p={p} ===")
        result = solve_uncapacitated_p_median(demand_col, p, candidate_pairs, lsoa_master,
                                               time_limit_sec=TIME_LIMIT_SEC, msg=False)
        if result["status"] == "Optimal":
            metrics = compute_metrics(result["dist"], lsoa_master)
        else:
            metrics = {"M1_avg_dist_m": None, "M2_coverage_800m": None, "M3_imd_gap": None, "M4_gini": None}
        core_results[key] = {**result, **metrics}
        core_summary_rows.append({
            "scenario": alpha_label, "p": p, "status": result["status"],
            "objective": result["objective"], "runtime_sec": round(result["runtime_sec"], 1),
            **metrics,
        })
        print(f"  -> status={result['status']}, runtime={result['runtime_sec']:.1f}s")

core_summary = pd.DataFrame(core_summary_rows)
print()
print("=== Core grid: M1-M4 for all 12 configurations ===")
print(core_summary.to_string(index=False))

## 5. Save results

In [ ]:
results_wide = lsoa_master[["lsoa_code"]].copy()
for (alpha_label, p), result in core_results.items():
    results_wide[f"zj_{alpha_label}_p{p}"] = result["zj"] if result["zj"] is not None else np.nan

output_path = os.path.join(BASE, "05_processed/uncapacitated_pmedian_results.csv")
results_wide.to_csv(output_path, index=False)
print(f"Saved per-LSOA zj (12 configurations): {output_path}")

core_summary.to_csv(os.path.join(BASE, "06_outputs/uncapacitated_core_grid_summary.csv"), index=False)
print("Saved: uncapacitated_core_grid_summary.csv")

## 6. Diagnostics — equity mechanism check

Same H1/H2-style check as the other two notebooks: does the correlation between site
selection (zj) and income_score shift toward positive as alpha increases?

In [ ]:
print("=== Pearson r: zj (site opened) vs income_score, by scenario (p=250) ===")
diag_table = results_wide.merge(lsoa_master[["lsoa_code", "income_score"]], on="lsoa_code", how="left")
for alpha_label in ["A", "B", "C", "D"]:
    col = f"zj_{alpha_label}_p250"
    if diag_table[col].notna().all():
        r, p_val = pearsonr(diag_table[col], diag_table["income_score"])
        print(f"Scenario {alpha_label}: r = {r:.4f}, p = {p_val:.4g}")
    else:
        print(f"Scenario {alpha_label}: not available (solve did not reach Optimal)")

## 7. Comparison note

This notebook's M1-M4 (no capacity, no eⱼ, no K) is the benchmark against which
05b_two_stage_model.ipynb (adds eⱼ + K + capacity) can be read as an ablation: does
bringing in existing supply and a capacity budget change the conclusions, or just add
detail? 05_p_median_715.ipynb (joint MILP + slack) shows why a literal joint capacitated
version is not solvable in a meaningful way at this scale.